# Agent device

Simulates the device carried by a rescue agent (an ambulance, say).

Unlike a subject device, an agent device transmits **all the time**, so the agent
stays visible on the emergency center's map. What it sends is only *recorded*
while the agent is on a mission.

It also listens. The center dispatches an agent by sending an assignment down
this same socket, and the device has to answer:

1. the center dispatches  ->  `mission_assigned` arrives here
2. this notebook replies  ->  `mission_ack`, which is what puts the mission *in progress*
3. the agent drives to the subject (the telemetry loop below)
4. this notebook reports  ->  `mission_report` with an outcome
5. the center finalises   ->  `mission_closed` arrives here

**Run this notebook before the subject one.** An agent that has not transmitted
recently counts as offline and will not be offered for dispatch, so the alert
would have nobody to route to.

In [ ]:
import websocket
import threading
import time
from datetime import datetime
import json

In [ ]:
mode = 'production'  # You can change to developement

In [ ]:
ws = websocket.WebSocket()
base = 'ws://localhost:8000' if mode == 'developement' else 'ws://localhost'

api_key_agent = "pbkdf2_sha256$720000$DFutwXSTzw15k2bXomDVyI$qROrV6SZr7oxvQbSI6S25LyxCswKkLXAuUQ2JLZ3Ac8="
ws.connect(f'{base}/ws/agent/send/', header={'device' : f'Resc {api_key_agent}', })

# So the listener below can give up waiting and check whether it should stop,
# rather than blocking on recv() for ever.
ws.settimeout(1)

# What the listener has learnt from the server. The mission id is what the
# acknowledgement and the report both have to quote.
state = {'mission': None, 'listening': True}


def acknowledge(mission_id) :
    """Confirm the assignment. This is what starts the mission."""
    ws.send(json.dumps({'type': 'mission_ack', 'mission_id': mission_id}))


def listen() :
    while state['listening'] :
        try :
            raw = ws.recv()
        except websocket.WebSocketTimeoutException :
            continue
        except Exception :
            return
        if not raw :
            return

        message = json.loads(raw)

        if 'mission_assigned' in message :
            mission = message['mission_assigned']
            state['mission'] = mission
            print(f"assigned mission {mission['id']}, subject at {mission['lat']}, {mission['lng']}")
            # A real device would confirm as soon as the crew accepted it.
            acknowledge(mission['id'])

        elif 'mission_closed' in message :
            print(f"mission {message['mission_closed']['id']} was closed by the center")
            state['mission'] = None

        elif 'mission' in message :
            print(f"mission {message['mission']['id']} is now at status {message['mission']['status']}")

        elif 'error' in message :
            print('error:', message['error'])


threading.Thread(target=listen, daemon=True).start()
print('connected, waiting for an assignment')

Send one position straight away, so the agent is on the map and counts as live.
Until a device has reported a position it cannot be ranked by distance, and so
cannot be dispatched.

In [ ]:
lattitude, longitude = 44.481247, 11.366309

ws.send(json.dumps({
    "lat" : lattitude,
    "lng" : longitude,
    "timestamp" : str(datetime.now().replace(microsecond=0)),
}))
print('position sent; this agent can now be dispatched')

Now start the **subject** notebook, and dispatch this agent from the center
dashboard. The assignment will be printed above by the listener.

Then run the cell below to drive to the subject. The path ends at the subject's
location. Note it no longer closes the socket at the end: the mission is still
open, the readings are being recorded against it, and the outcome still has to
be reported.

In [ ]:
agent_path = [(11.366309, 44.481247), (11.364764, 44.481675), (11.365193, 44.482441), (11.363906, 44.482839), (11.364335, 44.483543), (11.365665, 44.483329), (11.366695, 44.48489), (11.36558, 44.48538), (11.366095, 44.486391), (11.366223, 44.487829), (11.366138, 44.488687), (11.365623, 44.489115), (11.365108, 44.488472), (11.36455, 44.487676), (11.363734, 44.487676), (11.362833, 44.487983), (11.362018, 44.488289), (11.36146, 44.488442), (11.360601, 44.48884), (11.359743, 44.489146), (11.358928, 44.489452), (11.357941, 44.489758), (11.357563, 44.488766), (11.357906, 44.487358), (11.357906, 44.486317), (11.357906, 44.485582), (11.357855, 44.485564), (11.357211, 44.485319), (11.356653, 44.48486), (11.356267, 44.48437), (11.355323, 44.484462), (11.354422, 44.484737), (11.35352, 44.484737), (11.352877, 44.484829), (11.351975, 44.48486), (11.351117, 44.485074), (11.350087, 44.485196), (11.349315, 44.485319), (11.348328, 44.485503), (11.347426, 44.485748), (11.346396, 44.485992), (11.345452, 44.486513), (11.344937, 44.486911), (11.344551, 44.487278), (11.343907, 44.487523), (11.343307, 44.487646), (11.342534, 44.487768), (11.342362, 44.487187), (11.341504, 44.487309)]

try :
    for point in agent_path :

        longitude, lattitude = point
        data = {
            "lat" : lattitude,
            "lng" : longitude,
            "timestamp" : str(datetime.now().replace(microsecond=0))
        }
        ws.send(json.dumps(data))
        time.sleep(2)

    print('arrived at the subject')

except KeyboardInterrupt :
    print('stopped on the way')

except Exception as e:
    print(f"Error occurred: {e}")

Report how it went. This does **not** close the mission: the center still has to
finalise it, and may record a different outcome from the one reported here.

`1` is success, `2` is failure. Calling a mission off is the center's decision,
not the agent's, so there is no third option here.

In [ ]:
outcome = 1  # 1 = SUCCESS, 2 = FAILURE

mission = state['mission']

if mission is None :
    print('no mission in progress')
else :
    ws.send(json.dumps({
        'type': 'mission_report',
        'mission_id': mission['id'],
        'outcome': outcome,
    }))
    print(f"reported outcome {outcome}; the center still has to finalise mission {mission['id']}")

Finally, disconnect. Anything still buffered is flushed on the way out, so the
tail of the mission's track is not lost.

In [ ]:
state['listening'] = False
time.sleep(1.5)
ws.close()
print("WebSocket connection closed.")